In [1]:
# ============================================================
# LATE-ARRIVING FEEDBACK - MVP
#
# מקור:
# demo.bronze.learning_feedback
#
# כלל:
# feedback שמגיע עד 48 שעות לאחר feedback_time מתקבל לעיבוד.
# feedback שמגיע לאחר יותר מ-48 שעות מסומן TOO_LATE.
#
# בשלב הזה לא יוצרים טבלאות חדשות:
# - הנתונים התקינים כבר עוברים לטבלאות Silver הקיימות
# - אירועים מאוחרים מדי מוצגים לבקרה
# ============================================================

from pyspark.sql.functions import (
    col,
    when,
    round as spark_round,
    count,
    lit
)

print("=" * 90)
print("LATE-ARRIVING FEEDBACK CHECK - MVP")
print("=" * 90)


# ------------------------------------------------------------
# 1. Read the existing Bronze feedback table
# ------------------------------------------------------------

bronze_feedback_df = spark.table(
    "demo.bronze.learning_feedback"
)

print("Bronze feedback rows:", bronze_feedback_df.count())

bronze_feedback_df.printSchema()


# ------------------------------------------------------------
# 2. Calculate arrival delay
#
# feedback_time  = when the learner submitted the feedback
# ingestion_time = when the data platform received it
# ------------------------------------------------------------

feedback_with_delay_df = (
    bronze_feedback_df
    .withColumn(
        "delay_minutes_calculated",
        (
            col("ingestion_time").cast("long")
            - col("feedback_time").cast("long")
        ) / 60.0
    )
    .withColumn(
        "delay_hours_calculated",
        col("delay_minutes_calculated") / 60.0
    )
)


# ------------------------------------------------------------
# 3. Apply the 48-hour handling rule
# ------------------------------------------------------------

classified_feedback_df = (
    feedback_with_delay_df
    .withColumn(
        "late_arrival_status",
        when(
            col("feedback_time").isNull()
            | col("ingestion_time").isNull(),
            lit("INVALID_TIMESTAMP")
        )
        .when(
            col("delay_minutes_calculated") < 0,
            lit("INVALID_NEGATIVE_DELAY")
        )
        .when(
            col("delay_hours_calculated") <= 48,
            lit("ACCEPTED")
        )
        .otherwise(
            lit("TOO_LATE")
        )
    )
)


# ------------------------------------------------------------
# 4. Separate accepted and too-late feedback
# ------------------------------------------------------------

accepted_feedback_df = (
    classified_feedback_df
    .filter(
        col("late_arrival_status") == "ACCEPTED"
    )
)

too_late_feedback_df = (
    classified_feedback_df
    .filter(
        col("late_arrival_status") == "TOO_LATE"
    )
)

invalid_feedback_df = (
    classified_feedback_df
    .filter(
        col("late_arrival_status").isin(
            "INVALID_TIMESTAMP",
            "INVALID_NEGATIVE_DELAY"
        )
    )
)


# ------------------------------------------------------------
# 5. Display classification summary
# ------------------------------------------------------------

print("\nLate-arrival classification summary:")

classified_feedback_df.groupBy(
    "late_arrival_status"
).agg(
    count("*").alias("row_count")
).orderBy(
    "late_arrival_status"
).show(truncate=False)


# ------------------------------------------------------------
# 6. Display all feedback with delay
# ------------------------------------------------------------

print("Feedback delay details:")

classified_feedback_df.select(
    "feedback_id",
    "user_id",
    "session_id",
    "practice_id",
    "feedback_stage",
    "feedback_time",
    "ingestion_time",

    spark_round(
        col("delay_minutes_calculated"),
        2
    ).alias("delay_minutes"),

    spark_round(
        col("delay_hours_calculated"),
        2
    ).alias("delay_hours"),

    "late_arrival_status"
).orderBy(
    "ingestion_time"
).show(
    truncate=False
)


# ------------------------------------------------------------
# 7. Verify existing Silver tables
#
# Silver already stores delay_minutes.
# We verify that accepted Bronze feedback was transformed.
# ------------------------------------------------------------

silver_feedback_tables = [
    "demo.silver.pre_practice_feedback",
    "demo.silver.post_practice_feedback",
    "demo.silver.learner_check_in"
]

print("\nExisting Silver feedback tables:")

for table_name in silver_feedback_tables:
    silver_df = spark.table(table_name)

    print(
        table_name,
        "rows =",
        silver_df.count()
    )

    silver_df.select(
        "feedback_id",
        "feedback_time",
        "ingestion_time",
        "delay_minutes"
    ).orderBy(
        "feedback_time"
    ).show(
        truncate=False
    )


# ------------------------------------------------------------
# 8. MVP validation
# ------------------------------------------------------------

accepted_count = accepted_feedback_df.count()
too_late_count = too_late_feedback_df.count()
invalid_count = invalid_feedback_df.count()

total_classified = (
    accepted_count
    + too_late_count
    + invalid_count
)

source_count = bronze_feedback_df.count()

print("=" * 90)
print("LATE ARRIVAL MVP RESULT")
print("=" * 90)

print("Source feedback rows:", source_count)
print("Accepted within 48 hours:", accepted_count)
print("Too late - over 48 hours:", too_late_count)
print("Invalid timestamps/delays:", invalid_count)

if total_classified == source_count:
    print("Classification completeness: PASS")
else:
    print("Classification completeness: FAIL")

if invalid_count == 0:
    print("Timestamp quality: PASS")
else:
    print("Timestamp quality: WARNING")

print("\nHandling rule:")
print("0-48 hours  -> accepted and processed into existing Silver feedback tables")
print(">48 hours   -> rejected from normal processing and flagged as TOO_LATE")
print("Invalid time -> flagged for quality investigation")

print("=" * 90)

LATE-ARRIVING FEEDBACK CHECK - MVP
Bronze feedback rows: 9
root
 |-- feedback_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- practice_id: string (nullable = true)
 |-- feedback_stage: string (nullable = true)
 |-- feedback_time: timestamp (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- raw_payload: string (nullable = true)


Late-arrival classification summary:


+-------------------+---------+
|late_arrival_status|row_count|
+-------------------+---------+
|ACCEPTED           |9        |
+-------------------+---------+

Feedback delay details:
+------------+--------+-----------+------------+----------------+-------------------+-------------------+-------------+-----------+-------------------+
|feedback_id |user_id |session_id |practice_id |feedback_stage  |feedback_time      |ingestion_time     |delay_minutes|delay_hours|late_arrival_status|
+------------+--------+-----------+------------+----------------+-------------------+-------------------+-------------+-----------+-------------------+
|feedback_001|user_001|session_001|practice_001|before_practice |2026-07-20 09:04:00|2026-07-20 09:04:03|0.05         |0.0        |ACCEPTED           |
|feedback_002|user_001|session_001|practice_001|after_practice  |2026-07-20 09:18:00|2026-07-20 09:22:00|4.0          |0.07       |ACCEPTED           |
|feedback_003|user_002|session_002|practice_002|before_

In [2]:
from datetime import datetime, timedelta
from pyspark.sql.functions import col, lit, when

now = datetime.now()
late_feedback_time = now - timedelta(hours=72)

late_test_df = spark.createDataFrame(
    [
        (
            "feedback_late_test",
            "user_001",
            "session_late_test",
            "practice_late_test",
            "after_practice",
            late_feedback_time,
            now,
            '{"test_record": true}'
        )
    ],
    schema=bronze_feedback_df.schema
)

late_arrival_test_df = (
    bronze_feedback_df
    .unionByName(late_test_df)
    .withColumn(
        "delay_minutes_calculated",
        (
            col("ingestion_time").cast("long")
            - col("feedback_time").cast("long")
        ) / 60.0
    )
    .withColumn(
        "delay_hours_calculated",
        col("delay_minutes_calculated") / 60.0
    )
    .withColumn(
        "late_arrival_status",
        when(
            col("feedback_time").isNull()
            | col("ingestion_time").isNull(),
            lit("INVALID_TIMESTAMP")
        )
        .when(
            col("delay_minutes_calculated") < 0,
            lit("INVALID_NEGATIVE_DELAY")
        )
        .when(
            col("delay_hours_calculated") <= 48,
            lit("ACCEPTED")
        )
        .otherwise(
            lit("TOO_LATE")
        )
    )
)

late_arrival_test_df.filter(
    col("feedback_id") == "feedback_late_test"
).select(
    "feedback_id",
    "feedback_time",
    "ingestion_time",
    "delay_hours_calculated",
    "late_arrival_status"
).show(truncate=False)

+------------------+--------------------------+--------------------------+----------------------+-------------------+
|feedback_id       |feedback_time             |ingestion_time            |delay_hours_calculated|late_arrival_status|
+------------------+--------------------------+--------------------------+----------------------+-------------------+
|feedback_late_test|2026-07-26 10:02:45.871891|2026-07-29 10:02:45.871891|72.0                  |TOO_LATE           |
+------------------+--------------------------+--------------------------+----------------------+-------------------+

